# Data standards: FHIR, HL7, ICD, SNOMED CT, LOINC and OMOP

Exercise notebook for the lesson **[Data standards: FHIR, HL7, ICD, SNOMED CT, LOINC and OMOP](https://ramkumarmd.com/medical-ai/foundations/healthcare-data-standards)** in the *Foundations* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** Healthcare data standards fall into two groups. Exchange standards such as HL7 v2 and FHIR define how data moves between systems; terminologies such as ICD, SNOMED CT and LOINC define what each item means. Research models such as OMOP put data from many sources into one shape. AI projects that ignore these standards spend most of their time reconciling codes and units by hand.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Exchange: HL7 v2 and FHIR

Read the test, value and units from a synthetic FHIR R4 Observation.

In [ ]:
import json

observation_json = """
{
  "resourceType": "Observation",
  "status": "final",
  "code": {"coding": [{"system": "http://loinc.org", "code": "4548-4",
            "display": "Hemoglobin A1c/Hemoglobin.total in Blood"}]},
  "subject": {"reference": "Patient/example-001"},
  "effectiveDateTime": "2026-03-14",
  "valueQuantity": {"value": 7.8, "unit": "%",
                    "system": "http://unitsofmeasure.org", "code": "%"}
}
"""
obs = json.loads(observation_json)

coding = obs["code"]["coding"][0]
quantity = obs["valueQuantity"]
print(f"{coding['display']} (LOINC {coding['code']})")
print(f"{quantity['value']} {quantity['unit']} on {obs['effectiveDateTime']}")

# Hemoglobin A1c/Hemoglobin.total in Blood (LOINC 4548-4)
# 7.8 % on 2026-03-14

### Harmonising codes and units across sites

Map local codes to LOINC and convert units so both sites' results line up.

In [ ]:
LOCAL_TO_LOINC = {
    ("hospital_a", "GLU"): "2345-7",      # Glucose [Mass/volume] in Serum or Plasma
    ("hospital_b", "GLUC_P"): "2345-7",
    ("hospital_a", "A1C"): "4548-4",      # Hemoglobin A1c/Hemoglobin.total in Blood
    ("hospital_b", "HBA1C_IFCC"): "4548-4",
}

def harmonise(source, local_code, value, unit):
    loinc = LOCAL_TO_LOINC[(source, local_code)]
    if loinc == "2345-7" and unit == "mmol/L":
        value, unit = value * 18.0, "mg/dL"          # glucose: 1 mmol/L ≈ 18 mg/dL
    if loinc == "4548-4" and unit == "mmol/mol":
        value, unit = value / 10.929 + 2.15, "%"     # IFCC → NGSP master equation
    return loinc, round(value, 1), unit

results = [
    ("hospital_a", "GLU", 126, "mg/dL"),
    ("hospital_b", "GLUC_P", 7.0, "mmol/L"),
    ("hospital_a", "A1C", 7.0, "%"),
    ("hospital_b", "HBA1C_IFCC", 53, "mmol/mol"),
]
for r in results:
    print(r, "->", harmonise(*r))

# ('hospital_a', 'GLU', 126, 'mg/dL') -> ('2345-7', 126, 'mg/dL')
# ('hospital_b', 'GLUC_P', 7.0, 'mmol/L') -> ('2345-7', 126.0, 'mg/dL')
# ('hospital_a', 'A1C', 7.0, '%') -> ('4548-4', 7.0, '%')
# ('hospital_b', 'HBA1C_IFCC', 53, 'mmol/mol') -> ('4548-4', 7.0, '%')

## Exercise

**Goal:** Read synthetic FHIR R4 patient bundles from Synthea, extract HbA1c and glucose Observations by LOINC code, and harmonise their units into one analysis table.

**Data:** [Synthea synthetic FHIR R4 bundles](https://github.com/synthetichealth/synthea) (Synthea is Apache 2.0; generated records are synthetic)

### Step 1

Load a few Synthea FHIR bundles (the notebook downloads a small sample) and count the resources of each type.

In [ ]:
# Your code here


### Step 2

Extract every Observation with LOINC 4548-4 or 2345-7 into a pandas table with patient, date, value and unit.

In [ ]:
# Your code here


### Step 3

Add a second, made-up 'site' whose results use mmol/L and mmol/mol, and harmonise both sites with the mapping approach above.

In [ ]:
# Your code here


### Step 4

Look up 44054006 in a SNOMED CT browser and E11 / 5A11 in the WHO ICD browsers, and note how the concepts differ.

In [ ]:
# Your code here


### Step 5

Sketch which OMOP tables (person, measurement, condition_occurrence) each extracted item would go into.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/foundations/healthcare-data-standards](https://ramkumarmd.com/medical-ai/foundations/healthcare-data-standards). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.